In [1]:
import pandas as pd

In [2]:
FoodDataCleaned = pd.read_csv("FoodBank_cleaned.csv")

In [3]:
LIHEAPDataCleaned = pd.read_csv("LIHEAP_validated.csv")

In [4]:
CalFreshDataCleaned = pd.read_csv("CalFresh_cleaned.csv")

In [5]:
FoodDataCleaned.head(5)

,Zip Code,Month Year,Pounds_Distributed,Source,Pounds_Distributed_Outlier,Any_Outlier
0,91901,2018-01,7884.0,FSD,False,False
1,91901,2018-02,10558.0,FSD,False,False
2,91901,2018-03,10642.0,FSD,False,False
3,91901,2018-04,12192.0,FSD,False,False
4,91901,2018-05,10370.0,FSD,False,False


In [6]:
LIHEAPDataCleaned.head(5)

,Zip Code,Month Year,Enrollment_Count,Is_Negative,Outlier
0,91901,2023-02,1,False,False
1,91901,2023-03,2,False,False
2,91901,2023-04,1,False,False
3,91901,2023-05,2,False,False
4,91901,2023-06,2,False,False


In [7]:
CalFreshDataCleaned.head(5)

,Date,Zip Code,Calfresh Dollars,Meal Cost,Calfresh Meals,Month Year,Calfresh Dollars_Outlier,Meal Cost_Outlier,Calfresh Meals_Outlier,Any_Outlier
0,2020-01-01,91901,101567.0,3.473,29244.742330,2020-01,False,False,False,False
1,2020-01-01,91902,88881.0,3.473,25591.992910,2020-01,False,False,False,False
2,2020-01-01,91903,8938.0,3.473,2573.567271,2020-01,False,False,False,False
3,2020-01-01,91905,37603.0,3.473,10827.237650,2020-01,False,False,False,False
4,2020-01-01,91906,64151.0,3.473,18471.348620,2020-01,False,False,False,False


In [8]:
san_diego_zips_standardized = [
    "91901", "91902", "91903", "91905", "91906", "91908", "91909", "91910",
    "91911", "91912", "91913", "91914", "91915", "91916", "91917", "91921",
    "91931", "91932", "91933", "91934", "91935", "91941", "91942", "91943",
    "91944", "91945", "91946", "91950", "91951", "91962", "91963", "91976",
    "91977", "91978", "91979", "91980", "92003", "92004", "92007", "92008",
    "92009", "92010", "92011", "92013", "92014", "92018", "92019", "92020",
    "92021", "92022", "92023", "92024", "92025", "92026", "92027", "92028",
    "92029", "92030", "92033", "92036", "92037", "92038", "92039", "92040",
    "92046", "92049", "92051", "92052", "92054", "92055", "92056", "92057",
    "92058", "92059", "92060", "92061", "92064", "92065", "92066", "92067",
    "92068", "92069", "92070", "92071", "92072", "92074", "92075", "92078",
    "92079", "92081", "92082", "92083", "92084", "92085", "92086", "92088",
    "92091", "92092", "92093", "92101", "92102", "92103", "92104", "92105",
    "92106", "92107", "92108", "92109", "92110", "92111", "92112", "92113",
    "92114", "92115", "92116", "92117", "92118", "92119", "92120", "92121",
    "92122", "92123", "92124", "92126", "92127", "92128", "92129", "92130",
    "92131", "92136", "92137", "92138", "92139", "92140", "92142", "92143",
    "92145", "92149", "92150", "92153", "92154", "92159", "92160", "92163",
    "92165", "92166", "92167", "92168", "92169", "92170", "92171", "92172",
    "92173", "92174", "92175", "92176", "92177", "92178", "92186", "92191",
    "92192", "92193", "92195", "92196", "92198", 
]

In [9]:

def month_range(df):
    return df["Month Year"].min(), df["Month Year"].max()

food_min, food_max = month_range(FoodDataCleaned)
liheap_min, liheap_max = month_range(LIHEAPDataCleaned)
calfresh_min, calfresh_max = month_range(CalFreshDataCleaned)

overlap_start = max(food_min, liheap_min, calfresh_min)
overlap_end = min(food_max, liheap_max, calfresh_max)

print(f"Overlap window: {overlap_start} to {overlap_end}")

overlap_months = pd.period_range(overlap_start, overlap_end, freq='M').astype(str)

food_keys = set(zip(FoodDataCleaned['Zip Code'].astype(str).str.zfill(5),
                     FoodDataCleaned['Month Year']))
liheap_keys = set(zip(LIHEAPDataCleaned['Zip Code'].astype(str).str.zfill(5),
                       LIHEAPDataCleaned['Month Year']))
calfresh_keys = set(zip(CalFreshDataCleaned['Zip Code'].astype(str).str.zfill(5),
                         CalFreshDataCleaned['Month Year']))

scaffold = pd.MultiIndex.from_product(
    [san_diego_zips_standardized, overlap_months],
    names=['Zip Code', 'Month Year']
).to_frame(index=False)

keys = list(zip(scaffold['Zip Code'], scaffold['Month Year']))

coverage = scaffold.copy()
coverage['Has_FoodBank'] = [k in food_keys for k in keys]
coverage['Has_LIHEAP'] = [k in liheap_keys for k in keys]
coverage['Has_CalFresh'] = [k in calfresh_keys for k in keys]
coverage['All_Three'] = coverage['Has_FoodBank'] & coverage['Has_LIHEAP'] & coverage['Has_CalFresh']

Overlap window: 2023-01 to 2023-07


In [10]:
print(calfresh_min, calfresh_max)

2020-01 2025-08


In [11]:
print(liheap_min, liheap_max)

2023-01 2023-07


In [12]:
print(food_min, food_max)

2018-01 2025-08


In [13]:
coverage

,Zip Code,Month Year,Has_FoodBank,Has_LIHEAP,Has_CalFresh,All_Three
0,91901,2023-01,True,False,True,False
1,91901,2023-02,True,True,True,True
2,91901,2023-03,True,True,True,True
3,91901,2023-04,True,True,True,True
4,91901,2023-05,True,True,True,True
...,...,...,...,...,...,...
1150,92198,2023-03,False,False,True,False
1151,92198,2023-04,False,True,True,False
1152,92198,2023-05,False,False,True,False
1153,92198,2023-06,False,False,True,False


In [34]:
coverage["Zip Code"].nunique()

165

In [14]:
coverage.to_csv('TemporalCoverage_Matrix.csv', index=False)

**Generate Zip Code Coverage Across All Datasets**

In [15]:
all_keys = sorted(food_keys | liheap_keys | calfresh_keys)

In [16]:
zip_month_coverage = pd.DataFrame(all_keys, columns=["Zip Code", "Month Year"])
zip_month_coverage["Present in CalFresh"] = [k in calfresh_keys for k in all_keys]
zip_month_coverage["Present in LIHEAP"] = [k in liheap_keys for k in all_keys]
zip_month_coverage["Present in FoodBank"] = [k in food_keys for k in all_keys]

print(f"Total zip/month rows: {len(zip_month_coverage)}")
not_in_all_three = (~zip_month_coverage[['Present in CalFresh', 'Present in LIHEAP', 'Present in FoodBank']].all(axis=1)).sum()
print(f"Rows missing from at least one source: {not_in_all_three}")
zip_month_coverage

Total zip/month rows: 13598
Rows missing from at least one source: 13072


,Zip Code,Month Year,Present in CalFresh,Present in LIHEAP,Present in FoodBank
0,91901,2018-01,False,False,True
1,91901,2018-02,False,False,True
2,91901,2018-03,False,False,True
3,91901,2018-04,False,False,True
4,91901,2018-05,False,False,True
...,...,...,...,...,...
13593,92198,2025-04,True,False,False
13594,92198,2025-05,True,False,False
13595,92198,2025-06,True,False,False
13596,92198,2025-07,True,False,False


In [17]:
 zip_month_coverage.to_csv('Zip_Month_Coverage_Matrix.csv', index=False)

In [18]:
for name, df in [('FoodDataCleaned', FoodDataCleaned),
                  ('LIHEAPDataCleaned', LIHEAPDataCleaned),
                  ('CalFreshDataCleaned', CalFreshDataCleaned)]:
    col = df['Month Year']
    print(f"--- {name} ---")
    print("dtype:", col.dtype)
    print("nulls:", col.isna().sum())
    print("sample unique values (sorted):", sorted(col.dropna().unique())[:5], '...', sorted(col.dropna().unique())[-5:])
    print("min() / max():", col.min(), '/', col.max())
    print()

--- FoodDataCleaned ---
dtype: str
nulls: 0
sample unique values (sorted): ['2018-01', '2018-02', '2018-03', '2018-04', '2018-05'] ... ['2025-04', '2025-05', '2025-06', '2025-07', '2025-08']
min() / max(): 2018-01 / 2025-08

--- LIHEAPDataCleaned ---
dtype: str
nulls: 0
sample unique values (sorted): ['2023-01', '2023-02', '2023-03', '2023-04', '2023-05'] ... ['2023-03', '2023-04', '2023-05', '2023-06', '2023-07']
min() / max(): 2023-01 / 2023-07

--- CalFreshDataCleaned ---
dtype: str
nulls: 0
sample unique values (sorted): ['2020-01', '2020-02', '2020-03', '2020-04', '2020-05'] ... ['2025-04', '2025-05', '2025-06', '2025-07', '2025-08']
min() / max(): 2020-01 / 2025-08



**Merge the datasets together into one**
- Use zip code and Month Year as the common keys

In [19]:
liheap = pd.read_csv('LIHEAP_validated.csv')
calfresh = pd.read_csv('CalFresh_cleaned.csv')
foodbank = pd.read_csv('FoodBank_cleaned.csv')

In [20]:
for df in (liheap, calfresh, foodbank):
    df['Zip Code'] = df['Zip Code'].astype(str).str.zfill(5)

In [21]:
food_wide = (
    foodbank
    .pivot_table(index=['Zip Code', 'Month Year'], columns='Source',
                 values='Pounds_Distributed', aggfunc='sum')
    .reset_index()
)
food_wide.columns.name = None
food_wide = food_wide.rename(columns={'FSD': 'FSD_Pounds', 'SDFB': 'SDFB_Pounds'})
food_wide['Total_Pounds_Distributed'] = food_wide[['FSD_Pounds', 'SDFB_Pounds']].sum(axis=1, skipna=True)

food_outlier = foodbank.groupby(['Zip Code', 'Month Year'])['Any_Outlier'].any().reset_index()
food_wide = food_wide.merge(food_outlier, on=['Zip Code', 'Month Year'], how='left')
food_wide = food_wide.rename(columns={'Any_Outlier': 'FoodBank_Outlier'})

In [35]:
foodbank

,Zip Code,Month Year,Pounds_Distributed,Source,Pounds_Distributed_Outlier,Any_Outlier
0,91901,2018-01,7884.0,FSD,False,False
1,91901,2018-02,10558.0,FSD,False,False
2,91901,2018-03,10642.0,FSD,False,False
3,91901,2018-04,12192.0,FSD,False,False
4,91901,2018-05,10370.0,FSD,False,False
...,...,...,...,...,...,...
17199,92007,2025-04,0.0,SDFB,False,False
17200,92007,2025-05,0.0,SDFB,False,False
17201,92007,2025-06,0.0,SDFB,False,False
17202,92007,2025-07,0.0,SDFB,False,False


In [22]:
food_wide

,Zip Code,Month Year,FSD_Pounds,SDFB_Pounds,Total_Pounds_Distributed,FoodBank_Outlier
0,91901,2018-01,7884.0,0.000000,7884.000000,False
1,91901,2018-02,10558.0,0.000000,10558.000000,False
2,91901,2018-03,10642.0,0.000000,10642.000000,False
3,91901,2018-04,12192.0,0.000000,12192.000000,False
4,91901,2018-05,10370.0,0.000000,10370.000000,False
...,...,...,...,...,...,...
9379,92173,2025-04,1516.0,134556.197909,136072.197909,True
9380,92173,2025-05,997.0,134549.664942,135546.664942,True
9381,92173,2025-06,267.0,134440.021539,134707.021539,True
9382,92173,2025-07,1798.0,134430.789802,136228.789802,True


In [23]:
liheap_renamed = liheap.rename(columns={
    'Enrollment_Count': 'LIHEAP_Enrollment_Count',
    'Outlier': 'LIHEAP_Outlier'
})[['Zip Code', 'Month Year', 'LIHEAP_Enrollment_Count', 'LIHEAP_Outlier']]

calfresh_renamed = calfresh.rename(columns={
    'Any_Outlier': 'CalFresh_Outlier'
})[['Zip Code', 'Month Year', 'Calfresh Dollars', 'Meal Cost', 'Calfresh Meals', 'CalFresh_Outlier']]

In [24]:
liheap_renamed

,Zip Code,Month Year,LIHEAP_Enrollment_Count,LIHEAP_Outlier
0,91901,2023-02,1,False
1,91901,2023-03,2,False
2,91901,2023-04,1,False
3,91901,2023-05,2,False
4,91901,2023-06,2,False
...,...,...,...,...
573,92177,2023-06,1,False
574,92193,2023-04,1,False
575,92195,2023-05,1,False
576,92198,2023-01,1,False


In [25]:
calfresh_renamed

,Zip Code,Month Year,Calfresh Dollars,Meal Cost,Calfresh Meals,CalFresh_Outlier
0,91901,2020-01,101567.0,3.473000,29244.742330,False
1,91902,2020-01,88881.0,3.473000,25591.992910,False
2,91903,2020-01,8938.0,3.473000,2573.567271,False
3,91905,2020-01,37603.0,3.473000,10827.237650,False
4,91906,2020-01,64151.0,3.473000,18471.348620,False
...,...,...,...,...,...,...
10868,92192,2025-08,11205.0,4.657416,2405.840507,False
10869,92193,2025-08,190982.0,4.657416,41006.000150,False
10870,92195,2025-08,23563.0,4.657416,5059.243183,False
10871,92196,2025-08,14233.0,4.657416,3055.986429,False


In [26]:
merged = (
    calfresh_renamed
    .merge(liheap_renamed, on=['Zip Code', 'Month Year'], how='outer')
    .merge(food_wide, on=['Zip Code', 'Month Year'], how='outer')
)

In [27]:
merged

,Zip Code,Month Year,Calfresh Dollars,Meal Cost,Calfresh Meals,CalFresh_Outlier,LIHEAP_Enrollment_Count,LIHEAP_Outlier,FSD_Pounds,SDFB_Pounds,Total_Pounds_Distributed,FoodBank_Outlier
0,91901,2018-01,NaN,NaN,NaN,NaN,NaN,NaN,7884.0,0.0,7884.0,False
1,91901,2018-02,NaN,NaN,NaN,NaN,NaN,NaN,10558.0,0.0,10558.0,False
2,91901,2018-03,NaN,NaN,NaN,NaN,NaN,NaN,10642.0,0.0,10642.0,False
3,91901,2018-04,NaN,NaN,NaN,NaN,NaN,NaN,12192.0,0.0,12192.0,False
4,91901,2018-05,NaN,NaN,NaN,NaN,NaN,NaN,10370.0,0.0,10370.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...
13593,92198,2025-04,7966.02,4.625089,1722.349671,False,NaN,NaN,NaN,NaN,NaN,NaN
13594,92198,2025-05,9157.02,4.653009,1967.978089,False,NaN,NaN,NaN,NaN,NaN,NaN
13595,92198,2025-06,10087.03,4.631940,2177.711556,False,NaN,NaN,NaN,NaN,NaN,NaN
13596,92198,2025-07,9229.00,4.725154,1953.163959,False,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
# ---- Final naming and column order ----
merged = merged.rename(columns={
    'Calfresh Dollars': 'Calfresh_Dollars',
    'Meal Cost': 'Calfresh_Meal_Cost',
    'Calfresh Meals': 'Calfresh_Meals',
    "CalFresh_Outlier": "Calfresh_Outlier"
})
merged = merged[[
    'Zip Code', 'Month Year',
    'Calfresh_Dollars', 'Calfresh_Meal_Cost', 'Calfresh_Meals', 'Calfresh_Outlier',
    'LIHEAP_Enrollment_Count', 'LIHEAP_Outlier',
    'FSD_Pounds', 'SDFB_Pounds', 'Total_Pounds_Distributed', 'FoodBank_Outlier'
]]

In [29]:
merged

,Zip Code,Month Year,Calfresh_Dollars,Calfresh_Meal_Cost,Calfresh_Meals,Calfresh_Outlier,LIHEAP_Enrollment_Count,LIHEAP_Outlier,FSD_Pounds,SDFB_Pounds,Total_Pounds_Distributed,FoodBank_Outlier
0,91901,2018-01,NaN,NaN,NaN,NaN,NaN,NaN,7884.0,0.0,7884.0,False
1,91901,2018-02,NaN,NaN,NaN,NaN,NaN,NaN,10558.0,0.0,10558.0,False
2,91901,2018-03,NaN,NaN,NaN,NaN,NaN,NaN,10642.0,0.0,10642.0,False
3,91901,2018-04,NaN,NaN,NaN,NaN,NaN,NaN,12192.0,0.0,12192.0,False
4,91901,2018-05,NaN,NaN,NaN,NaN,NaN,NaN,10370.0,0.0,10370.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...
13593,92198,2025-04,7966.02,4.625089,1722.349671,False,NaN,NaN,NaN,NaN,NaN,NaN
13594,92198,2025-05,9157.02,4.653009,1967.978089,False,NaN,NaN,NaN,NaN,NaN,NaN
13595,92198,2025-06,10087.03,4.631940,2177.711556,False,NaN,NaN,NaN,NaN,NaN,NaN
13596,92198,2025-07,9229.00,4.725154,1953.163959,False,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
merged.to_csv("Assistance_Integrated.csv", index=False)

In [31]:
transformation_log_rows = [
    {
        'Dataset': 'FoodBank, LIHEAP, CalFresh (all)',
        'Field(s)': 'Zip Code',
        'Transformation': 'Cast to string, zero-padded to 5 digits (.str.zfill(5))',
        'Unit Conversion?': 'No',
        'Reason / Notes': 'Prevents join misses caused by integer-vs-string or dropped leading zeros before merging on Zip Code.'
    },
    {
        'Dataset': 'FoodBank',
        'Field(s)': 'Source (FSD, SDFB) -> FSD_Pounds, SDFB_Pounds',
        'Transformation': 'Pivoted long to wide on the Source column',
        'Unit Conversion?': 'No',
        'Reason / Notes': 'Source data has one row per (Zip Code, Month Year, Source); pivot gives one row per zip/month with separate columns for each source.'
    },
    {
        'Dataset': 'FoodBank',
        'Field(s)': 'FSD_Pounds, SDFB_Pounds -> Total_Pounds_Distributed',
        'Transformation': 'Row-wise sum, skipna=True',
        'Unit Conversion?': 'No',
        'Reason / Notes': 'A zip/month with only one source reporting still gets a non-null total instead of being treated as zero/missing for the other source. Pounds units unchanged, pure aggregation.'
    },
    {
        'Dataset': 'FoodBank',
        'Field(s)': 'Any_Outlier -> FoodBank_Outlier',
        'Transformation': 'Collapsed row-level flag to zip/month level via .any()',
        'Unit Conversion?': 'No',
        'Reason / Notes': 'If either source (FSD or SDFB) was flagged as an outlier for that zip/month, the combined row is flagged.'
    },
    {
        'Dataset': 'CalFresh',
        'Field(s)': 'Calfresh_Dollars, Calfresh_Meal_Cost, Calfresh_Meals, Calfresh_Outlier',
        'Transformation': 'Pass-through, column renamed only',
        'Unit Conversion?': 'No',
        'Reason / Notes': 'No reshaping or unit conversion applied; values used as-is from CalFresh_cleaned.csv.'
    },
    {
        'Dataset': 'LIHEAP',
        'Field(s)': 'LIHEAP_Enrollment_Count, LIHEAP_Outlier',
        'Transformation': 'Pass-through, column renamed only',
        'Unit Conversion?': 'No',
        'Reason / Notes': 'No reshaping or unit conversion applied; values used as-is from LIHEAP_validated.csv.'
    },
    {
        'Dataset': 'LIHEAP',
        'Field(s)': 'Enrollment_Count -> LIHEAP_Enrollment_Count',
        'Transformation': 'Column rename',
        'Unit Conversion?': 'No',
        'Reason / Notes': 'Namespaced to avoid ambiguity once merged with the other two datasets.'
    },
    {
        'Dataset': 'LIHEAP',
        'Field(s)': 'Outlier -> LIHEAP_Outlier',
        'Transformation': 'Column rename',
        'Unit Conversion?': 'No',
        'Reason / Notes': 'Namespaced to avoid ambiguity once merged with the other two datasets.'
    },
    {
        'Dataset': 'CalFresh',
        'Field(s)': 'Any_Outlier -> Calfresh_Outlier',
        'Transformation': 'Column rename',
        'Unit Conversion?': 'No',
        'Reason / Notes': 'Namespaced to avoid ambiguity once merged with the other two datasets.'
    },
    {
        'Dataset': 'CalFresh',
        'Field(s)': 'Calfresh Dollars -> Calfresh_Dollars',
        'Transformation': 'Column rename (space -> underscore)',
        'Unit Conversion?': 'No',
        'Reason / Notes': 'Naming consistency across the merged output.'
    },
    {
        'Dataset': 'CalFresh',
        'Field(s)': 'Meal Cost -> Calfresh_Meal_Cost',
        'Transformation': 'Column rename (space -> underscore, source-namespaced)',
        'Unit Conversion?': 'No',
        'Reason / Notes': 'Naming consistency across the merged output.'
    },
    {
        'Dataset': 'CalFresh',
        'Field(s)': 'Calfresh Meals -> Calfresh_Meals',
        'Transformation': 'Column rename (space -> underscore)',
        'Unit Conversion?': 'No',
        'Reason / Notes': 'Naming consistency across the merged output.'
    },
    {
        'Dataset': 'FoodBank',
        'Field(s)': 'FSD, SDFB -> FSD_Pounds, SDFB_Pounds',
        'Transformation': 'Column rename (post-pivot)',
        'Unit Conversion?': 'No',
        'Reason / Notes': 'Clarifies units (pounds) now that Source is no longer a labeled column after the pivot.'
    },
    {
        'Dataset': 'CalFresh, LIHEAP, FoodBank (merge)',
        'Field(s)': 'Zip Code, Month Year (join keys)',
        'Transformation': "Outer join across all three on ['Zip Code', 'Month Year']",
        'Unit Conversion?': 'No',
        'Reason / Notes': 'A zip/month present in only one source still appears in the merged output, with NaN for the datasets that had no record. Preserves visibility into coverage gaps rather than dropping them, consistent with the LIHEAP decision to retain gaps rather than impute/exclude.'
    },
]
 
transformation_log = pd.DataFrame(transformation_log_rows)
transformation_log

,Dataset,Field(s),Transformation,Unit Conversion?,Reason / Notes
0,"FoodBank, LIHEAP, CalFresh (all)",Zip Code,"Cast to string, zero-padded to 5 digits (.str....",No,Prevents join misses caused by integer-vs-stri...
1,FoodBank,"Source (FSD, SDFB) -> FSD_Pounds, SDFB_Pounds",Pivoted long to wide on the Source column,No,"Source data has one row per (Zip Code, Month Y..."
2,FoodBank,"FSD_Pounds, SDFB_Pounds -> Total_Pounds_Distri...","Row-wise sum, skipna=True",No,A zip/month with only one source reporting sti...
3,FoodBank,Any_Outlier -> FoodBank_Outlier,Collapsed row-level flag to zip/month level vi...,No,If either source (FSD or SDFB) was flagged as ...
4,CalFresh,"Calfresh_Dollars, Calfresh_Meal_Cost, Calfresh...","Pass-through, column renamed only",No,No reshaping or unit conversion applied; value...
5,LIHEAP,"LIHEAP_Enrollment_Count, LIHEAP_Outlier","Pass-through, column renamed only",No,No reshaping or unit conversion applied; value...
6,LIHEAP,Enrollment_Count -> LIHEAP_Enrollment_Count,Column rename,No,Namespaced to avoid ambiguity once merged with...
7,LIHEAP,Outlier -> LIHEAP_Outlier,Column rename,No,Namespaced to avoid ambiguity once merged with...
8,CalFresh,Any_Outlier -> Calfresh_Outlier,Column rename,No,Namespaced to avoid ambiguity once merged with...
9,CalFresh,Calfresh Dollars -> Calfresh_Dollars,Column rename (space -> underscore),No,Naming consistency across the merged output.


In [32]:
transformation_log.to_csv('Data_Transformation_Log.csv', index=False)